# PDC walkthrough — Proportion of Days Covered, on 2023 MEPS

**Today's goal:** turn the merged frame you already built in `allYearMerge.ipynb` into a real medication-adherence ratio — per patient per drug class per year. We are not redoing your work. We are picking up where it stops.

**What you've already done (and can defend):**
- Stacked 2020–2023 MEPS files with `meps_year` preserved.
- Joined h248a (Rx fills) → h248if1 (CLNK) → h249 (Conditions). That join graph is the structural thing most freshmen miss; you got it.
- Merged `is_chronic.xlsx` on `ICD10CDX` to flag chronic conditions.
- Dropped negative `RXDAYSUP` rows.
- Computed `groupby(["DUPERSID", "DRUGIDX"]).RXDAYSUP.sum()` for diabetes (E11). That gave you the mean ~340, max ~1600. **That number is real — it's the numerator of the adherence ratio, not the ratio itself.**

**What this notebook adds on top:**
1. A **denominator** — the number of days the patient was eligible to be taking the drug. This is the missing piece. Without it, the sum of days supplied is just an absolute count.
2. Two small cleanings you previously glossed over (`DIABEQUIP=1` supply rows and `RXDAYSUP=999` 'as needed' rows).
3. The maintenance-drug filter via Multum `TC1S1` (so PRN drugs do not pollute the ratio).
4. The full merge chain — h248a → CLNK → h249 → `is_chronic.xlsx` — done step-by-step here so every cell is yours.
5. **Two views of Sub-question 1:** PDC by drug class AND PDC by condition.

**Your three big sub-questions, for reference:**
1. Which medication groups show lower refill continuity? *(today, two views)*
2. (a) Does cost affect adherence? (b) How does chronic burden relate to adherence? (c) How do side effects relate to adherence? *(next sessions)*
3. Can we predict future non-adherence for a patient-drug pair? *(after the model)*

**How to read this notebook:** every code cell is preceded by plain-English explanation and followed by a 'wrestle with this' question. Doc links go to the official pandas / numpy / matplotlib / seaborn documentation — when you don't remember a method, click the link, don't guess.

**Two ground rules from your Cursor guide:**
- Before you run a cell, say out loud what you expect to see. The gap between guess and result is the lesson.
- If you can't explain a cell, it isn't yours yet. Mark it. We work through it together.

**Cursor rule for this session:** keep Cursor closed unless Mehak prompts it. The point of working through this notebook line by line is for you to internalize each step. We open Cursor only when (a) we don't understand a method or (b) the code keeps throwing the same error and we need a fresh angle. Otherwise, you read the doc link, you write the code yourself.

---
## Project flow — where this notebook sits in the whole project

This notebook is the **2023 single-year teaching reference** for the PDC calculation. The project has more pieces than just this. Here's the map so you always know what's coming.

**Stage 1 — Single-year PDC on 2023 (this notebook, today)**
- Section 1–13 below compute PDC per `(person, drug class)` and per `(person, drug class, condition)` for chronic patients on maintenance drugs in 2023 MEPS.
- Answers **Sub-question 1** in two views: by drug class and by condition.

**Stage 2 — You replicate this in your own 2023 notebook**
- Create `Notebooks/Friana_PDC_2023.ipynb`. Write the code yourself. Use this notebook as reference, not as something to copy from.
- Each cell you write, you should be able to explain without looking back here. That's how the code becomes yours, not Cursor's.

**Stage 3 — Expand to all years in `allYearMerge_v2.ipynb`**
- Once you're comfortable with the 2023 single-year version, we redo the same calculation on the stacked 2020–2023 frame.
- This is where year-suffix swaps come in: `h248a` → `h239a` → `h229a` → `h220a`, `RXSF23X` → `RXSF22X` → `RXSF21X` → `RXSF20X`, `PERWT23F` → `PERWT22F` → etc.
- The chronic flags on h251 (`DIABDX_M18`, `HIBPDX`, etc.) and the SAQ items (`DLAYPM42`) do NOT carry a year suffix; only the spending and weight variables do.

**Stage 4 — Sub-question 2a: cost**
- Section 14 below has the markdown stub. We code it in a future session.

**Stage 5 — Sub-question 2b: chronic burden**
- Section 15 below has the markdown stub.

**Stage 6 — Sub-question 2c: side effects**
- Section 16 below has the markdown stub. This is the one where MEPS hits a data wall; we'll talk about it then.

**Stage 7 — Sub-question 3: prediction (the model)**
- Section 17 below has the markdown stub for the modeling table, baselines, metric, and evaluation.

**Stage 8 — Python scripts (early August, integrated with the modeling work)**
- The notebook is the teaching artifact. The scripts are the reproducibility artifact. In early August we move each section of this notebook into a function in `src/med_adherence/` so the whole pipeline can be re-run end-to-end without a notebook open.
- Why early August and not mid-July: the modeling code in Section 18 is what most benefits from being in a script form (parameterized, repeatable training runs). Refactoring before the modeling exists means the scripts get rewritten when modeling lands; refactoring after means the scripts include modeling from day one.
- Mapping: Section 2 (cleaning) → `clean_h248a()`. Section 4 (CLNK filter) → `filter_clnk_pmed()`. Section 5–7 (the merge chain) → `merge_rx_with_conditions()`. Section 8 (chronic + maintenance) → `filter_chronic_maintenance()`. Section 9–10 (PDC) → `compute_pdc()`. Section 11–12 (views) → `plot_pdc_by_class()` / `plot_pdc_by_condition()`. Section 13 (h250) → `summarize_h250_to_person()`.
- The scripts will take a `year` argument and do the right thing for any of 2020/2021/2022/2023 — same logic, year-suffix swaps applied internally.

**Stage 9 — Streamlit front-end + GitHub**
- Mid-to-late August. Weekends 9 and 10 in your study plan. The model becomes a usable app.

**Stage 10 — Defense rehearsal**
- Weekend 11 (Aug 15-16). The whole story end-to-end.

---

**Year-suffix translation (memorize this — you'll use it constantly):**

| File family | 2020 | 2021 | 2022 | 2023 |
|---|---|---|---|---|
| Rx | h220a | h229a | h239a | h248a |
| CLNK | h220if1 | h229if1 | h239if1 | h248if1 |
| Conditions | h222 | h231 | h241 | h249 |
| Plans | h223 | h232 | h242 | h250 |
| Consolidated | h224 | h233 | h243 | h251 |

**Column-name year suffix:** `*23` → `*22` → `*21` → `*20` for spending and weight columns. So `RXSF23X` → `RXSF22X`, `RXSLF23` → `RXSLF22`, `PERWT23F` → `PERWT22F`, `AGE23X` → `AGE22X`. Chronic flags like `DIABDX_M18` and SAQ items like `DLAYPM42` do NOT carry a year suffix — they are the same name every year.

---
## Section 0 — Imports and file path

Same imports you've been using. The `calamine` engine reads MEPS `.xlsx` files about 10× faster than openpyxl. It's what your other notebooks already use.

**Docs to know:**
- pandas: https://pandas.pydata.org/docs/reference/index.html
- numpy: https://numpy.org/doc/stable/reference/index.html
- matplotlib pyplot: https://matplotlib.org/stable/api/pyplot_summary.html
- seaborn: https://seaborn.pydata.org/api.html
- MEPS HC-248A codebook (2023 Rx): https://meps.ahrq.gov/data_stats/download_data/pufs/h248a/h248acb.pdf
- MEPS HC-248I codebook (2023 CLNK): https://meps.ahrq.gov/data_stats/download_data/pufs/h248i/h248icb.pdf
- MEPS HC-249 codebook (2023 Conditions): https://meps.ahrq.gov/data_stats/download_data/pufs/h249/h249cb.pdf

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Make plots a bit larger by default so they're readable.
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Walk up from the current working directory and find the data/MEPS/excels folder.
# This is the same pattern your other notebooks use.
def resolve_meps_dir():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "MEPS",
        cwd.parent / "data" / "MEPS",
        cwd.parent.parent / "data" / "MEPS",
    ]
    for d in candidates:
        if (d / "excels" / "h248a.xlsx").exists():
            return d / "excels"
    raise FileNotFoundError("Could not find data/MEPS/excels")

MEPS = resolve_meps_dir()
print("Reading from:", MEPS)

---
## Section 1 — Load the four files we need

We need four files for the full PDC story:

| File | What's in it | One row = |
|---|---|---|
| `h248a.xlsx` | Prescription fills | one fill (~192k for 2023) |
| `h248if1.xlsx` | CLNK bridge — links fills to conditions | one event-condition link (~281k) |
| `h249.xlsx` | Medical conditions | one condition for a person (~63k) |
| `h250.xlsx` | Person Round Plan (private insurance plans) | one row per person × round × establishment × plan (~35k) |
| `is_chronic.xlsx` | Your hand-curated chronic flag per ICD-10 code | one ICD-10 code (~250) |

We load them one at a time so we can look at each before joining.

**Doc:** [pandas.read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)

**When you redo this for 2022, 2021, 2020:** the file names change but everything else stays the same. The mapping for the files we load here is:

| Year | Rx file | CLNK file | Conditions file | Plans file |
|---|---|---|---|---|
| 2023 | `h248a.xlsx` | `h248if1.xlsx` | `h249.xlsx` | `h250.xlsx` |
| 2022 | `h239a.xlsx` | `h239if1.xlsx` | `h241.xlsx` | `h242.xlsx` |
| 2021 | `h229a.xlsx` | `h229if1.xlsx` | `h231.xlsx` | `h232.xlsx` |
| 2020 | `h220a.xlsx` | `h220if1.xlsx` | `h222.xlsx` | `h223.xlsx` |

`is_chronic.xlsx` is the same file every year — it's a code-level lookup, not a year-specific dataset.

**h250 caveat:** 2023 is the final public release year for h250. AHRQ has discontinued it from 2024 onward. Use it for v1; document the cutoff.

**Porting to scripts (early August target — see closing section):** this whole section becomes `load_meps_year(year)` that returns five DataFrames. The function looks up the file names from a dict keyed by year and reads them. Same calamine engine, same columns.

### Load h248a (Prescription fills)

In [ ]:
h248a = pd.read_excel(MEPS / "h248a.xlsx", engine="calamine")
print("h248a shape:", h248a.shape)
print("h248a columns:", list(h248a.columns)[:10], "...")
h248a.head(3)

### Load h248if1 (CLNK — the bridge table)

The CLNK file is small — just 6 columns. It tells you which events are linked to which conditions for each person.

In [ ]:
h248if1 = pd.read_excel(MEPS / "h248if1.xlsx", engine="calamine")
print("h248if1 shape:", h248if1.shape)
print("h248if1 columns:", list(h248if1.columns))
h248if1.head(3)

### Load h249 (Medical conditions)

This is one row per condition per person. The condition is described by `ICD10CDX` — a 3-digit ICD-10 code (MEPS truncates the full code for confidentiality).

In [ ]:
h249 = pd.read_excel(MEPS / "h249.xlsx", engine="calamine")
print("h249 shape:", h249.shape)
print("h249 columns:", list(h249.columns)[:10], "...")
h249.head(3)

### Load h250 (Person Round Plan — private insurance)

h250 is plan-level, not person-level. One person can have multiple rows (one per insurance plan they're covered by). We'll summarize it to person-level later (Section 13) and merge it into the PDC frame.

**Important caveat:** many plan-design columns on h250 are -1 (inapplicable) for most rows because they apply only to specific plan types. `PMEDINS` (does the plan cover Rx?) is the cleanest variable. `OOPPREMX` and `ANNDEDCTP` are usable for a sub-cohort.

In [ ]:
h250 = pd.read_excel(MEPS / "h250.xlsx", engine="calamine")
print("h250 shape:", h250.shape)
print()
print("PMEDINS distribution (does the plan cover Rx?):")
print(h250["PMEDINS"].value_counts().sort_index())
print("  1 = covers Rx, 2 = does not, -7/-8 = missing")

### Load is_chronic.xlsx (your hand-curated chronic flag)

You built this file yourself by reading MEPS HC-249 Appendix 1 Table 1 and marking each 3-digit ICD-10 code as chronic (1) or not chronic (0). The sheet `Chronic_Flagged_Codes` has the 136 codes you flagged chronic.

In [ ]:
is_chronic = pd.read_excel(MEPS / "is_chronic.xlsx", engine="calamine", sheet_name="Table1_ICD10CDX")
print("is_chronic shape:", is_chronic.shape)
print("is_chronic columns:", list(is_chronic.columns))
is_chronic.head(5)

**Wrestle with this:** four files, four different unit-of-analysis levels. For each one, write down in one sentence what one row represents. Don't peek at the table above first — write your guess, then check.

---
## Section 2 — Clean the h248a sentinel codes

MEPS doesn't use NaN; it uses negative integers and `999` to mark different kinds of missing. pandas will sum these as real numbers if you forget. This is the source of most quietly-wrong adherence numbers.

**MEPS reserved codes you should know:**
- `-1` = Inapplicable (the question wasn't asked)
- `-7` = Refused
- `-8` = Don't Know
- `-14` = Not yet taken/used (RXBEGYRX only)
- `-15` = Cannot be computed
- `999` = Taken as needed (so 999 is NOT 999 days; it means PRN)

We work through this in three small steps. Don't skip ahead.

**Year invariance:** these sentinel codes are the same for every MEPS year. The cleaning code below works identically for 2020/2021/2022/2023. When you port this to a script, it becomes `clean_h248a(df)` and takes no `year` argument.

### Step 2.1 — Look at `RXDAYSUP` before cleaning anything

This is the messy-first method. Don't clean. Just look.

**Doc:** [Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html), [Series.describe](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html)

In [ ]:
print("RXDAYSUP summary:")
print(h248a["RXDAYSUP"].describe())
print()
print("How many rows have RXDAYSUP < 0 (sentinel codes)?")
n_negative = (h248a["RXDAYSUP"] < 0).sum()
print("  count:", n_negative)
print("  percent:", round(100 * n_negative / len(h248a), 1), "%")
print()
print("How many rows have RXDAYSUP == 999 ('as needed')?")
n_999 = (h248a["RXDAYSUP"] == 999).sum()
print("  count:", n_999)

**Wrestle with this:** roughly 27% of `RXDAYSUP` values are `-8` (Don't Know). Two honest options for handling this:
1. **Drop those rows.** Lose 27% of fills but everything that remains is real.
2. **Impute by drug-class median.** Atorvastatin has a clean 90-day median, so we'd put 90 in its missing rows. Keeps data, but we invent some of it.

**For today, we drop.** State this choice in your weaknesses paragraph. In 60 seconds, defend the drop over the impute.

### Step 2.2 — Drop sentinel and 999 rows from RXDAYSUP

We want to keep only values in the valid 1–990 range.

**Doc:** [Series.between](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html), [DataFrame.dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)

In [ ]:
# Make a copy so we don't change the original.
h248a_clean = h248a.copy()

# Mark out-of-range values as NaN (this is the 'mask values not in the good range' move).
rxdaysup_valid = h248a_clean["RXDAYSUP"].between(1, 990, inclusive="both")
h248a_clean.loc[~rxdaysup_valid, "RXDAYSUP"] = np.nan

# Then drop the NaN rows.
rows_before = len(h248a_clean)
h248a_clean = h248a_clean.dropna(subset=["RXDAYSUP"]).copy()
rows_after = len(h248a_clean)

print("Rows before:", rows_before)
print("Rows after dropping missing RXDAYSUP:", rows_after)
print("Lost:", rows_before - rows_after)

### Step 2.3 — Clean the other numeric columns we'll use

Same sentinel codes apply to `TC1`, `TC1S1`, `RXQUANTY`, `PURCHRD`. We replace negatives with NaN so they don't get summed by accident.

**Doc:** [Series.where](https://pandas.pydata.org/docs/reference/api/pandas.Series.where.html)

In [ ]:
# We do this with a for-loop so it's explicit, not a comprehension.
columns_to_clean = ["TC1", "TC1S1", "RXQUANTY", "PURCHRD"]

for col in columns_to_clean:
    # .where(condition) keeps the value where condition is True, sets NaN where False.
    is_valid = h248a_clean[col] >= 0
    h248a_clean[col] = h248a_clean[col].where(is_valid, other=np.nan)
    n_nan = h248a_clean[col].isna().sum()
    print(col, "NaN count after cleaning:", n_nan)

# RXDRGNAM is a string column. The masked drug sentinel is the literal string '-15'.
h248a_clean["RXDRGNAM"] = h248a_clean["RXDRGNAM"].replace("-15", np.nan)
print("RXDRGNAM masked-string count:", h248a_clean["RXDRGNAM"].isna().sum())

**Wrestle with this:** why did we use `Series.between` for `RXDAYSUP` but `Series.where(condition)` for the others?

(Hint: `between` is easier to read when you have a clean range. `where` is easier when the condition is more complex than a range. Same job, different ergonomics.)

---
## Section 3 — Drop `DIABEQUIP = 1` rows (diabetic supplies)

Here's a column you probably never noticed: `DIABEQUIP`. When it's `1`, the row is not a drug. It's a diabetic supply — test strips, lancets, glucose meters. MEPS hides the actual product name for confidentiality, which is why those rows have `RXDRGNAM == '-15'`.

If we leave them in, they inflate fill counts for diabetic patients without adding therapy meaning. They're also 7 of the 8 rows that had `RXDAYSUP = 999`. Excluding them is the cleanest move.

**Doc:** [DataFrame boolean indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)

**Year invariance:** the `DIABEQUIP` column exists in every MEPS year and the exclusion logic is identical. This is one line in the eventual `clean_h248a()` function.

In [ ]:
# How many supply rows are there?
is_supply = h248a_clean["DIABEQUIP"] == 1
n_supply = is_supply.sum()
print("DIABEQUIP=1 rows (supplies, not drugs):", n_supply)

# Drop them. The '~' means 'not' for boolean Series.
h248a_drugs = h248a_clean.loc[~is_supply].copy()
print("Rows after dropping supplies:", len(h248a_drugs))

**Wrestle with this:** if you had left supplies in and grouped by diabetic patients, would the per-patient sum of `RXDAYSUP` go up or down on average? Why does that matter for the adherence story?

---
## Section 4 — Filter CLNK to prescribed-medicine events

The CLNK file has rows for several types of events: office visits, ER visits, inpatient stays, home-health visits, and prescribed-medicine fills. They share the same table because the design intent is one bridge between every event type and every condition.

We only want the prescribed-medicine rows. The column `EVENTYPE` codes them as follows:

| EVENTYPE | Meaning | Approx % of CLNK rows in 2023 |
|---|---|---|
| 1 | Office-based visit | 51.8% |
| 2 | Outpatient department | 7.9% |
| 3 | Emergency room | 1.5% |
| 4 | Inpatient hospital | 0.8% |
| 7 | Home health | 3.2% |
| **8** | **Prescribed medicine — what we want** | **34.8%** |

If you forget this filter, every drug fill in the next merge picks up office visits and ER visits as if they were drug-related events. That's the silent contamination bug.

**Year invariance:** `EVENTYPE` codes are identical every year. `filter_clnk_pmed(clnk_df)` returns a frame of rows with EVENTYPE == 8. Same one-liner for every year.

In [ ]:
# Show the distribution so we understand what we're filtering.
print("EVENTYPE distribution in CLNK:")
print(h248if1["EVENTYPE"].value_counts().sort_index())
print()

# Filter to EVENTYPE = 8 (PMED only).
clnk_pmed = h248if1.loc[h248if1["EVENTYPE"] == 8].copy()
print("CLNK rows after EVENTYPE=8 filter:", len(clnk_pmed))

**Wrestle with this:** the CLNK file has 281,158 rows total. After filtering to `EVENTYPE = 8`, we have 97,941. But `h248a` has 192,275 fills. Why aren't there 192,275 CLNK rows? Why are there fewer?

(Hint: in MEPS, the same drug refilled multiple times in one round shares a single CLNK row because they share an `EVNTIDX`.)

---
## Section 5 — Merge h248a with CLNK

The bridge column on h248a is `LINKIDX`. The matching column on CLNK is `EVNTIDX`. (They mean the same thing — AHRQ uses different names on the two files for historical reasons.)

We also match on `DUPERSID` so we don't accidentally cross-match between people.

**Doc:** [DataFrame.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)

**Important:** use `how="inner"` here. That keeps only fills that have a matching condition link. About 4.2% of fills don't have a CLNK link at all — those are dropped silently. That's a known coverage gap; we mention it in the weaknesses paragraph.

**Year invariance:** the merge keys (`DUPERSID`, `LINKIDX`, `EVNTIDX`) are identical every year. `merge_rx_with_clnk(rx, clnk)` is a function that takes any year's cleaned frames and returns the merged result. No year argument needed.

In [ ]:
# Before merging, show the sizes.
print("h248a_drugs rows:", len(h248a_drugs))
print("clnk_pmed rows:", len(clnk_pmed))

# The merge.
rx_with_links = h248a_drugs.merge(
    clnk_pmed,
    left_on=["DUPERSID", "LINKIDX"],
    right_on=["DUPERSID", "EVNTIDX"],
    how="inner",
)

print("rx_with_links rows after merge:", len(rx_with_links))
print("That's {:.1f}% of h248a_drugs.".format(100 * len(rx_with_links) / len(h248a_drugs)))
rx_with_links.head(3)

**Wrestle with this:** the merged result has slightly more rows than `h248a_drugs` (the left side) had. That's strange — an inner merge usually shrinks the left side. Why did it grow here?

(Hint: one fill can link to multiple conditions, so a single h248a row can produce multiple output rows.)

---
## Section 6 — Merge with h249 to bring in the actual ICD-10 code

Now we know which CONDIDX each fill links to. We need the ICD-10 code, which lives in h249. Match on `DUPERSID + CONDIDX`.

**Year invariance:** `CONDIDX`, `ICD10CDX`, `CCSR1X`, and `AGEDIAG` exist on every year's conditions file with the same names. No translation needed.

In [ ]:
# We only need a few columns from h249 — pick them out so the result stays manageable.
h249_subset = h249[["DUPERSID", "CONDIDX", "ICD10CDX", "CCSR1X", "AGEDIAG"]].copy()
print("h249_subset rows:", len(h249_subset))

# Merge.
rx_with_conditions = rx_with_links.merge(
    h249_subset,
    on=["DUPERSID", "CONDIDX"],
    how="inner",
)

print("rx_with_conditions rows after merge:", len(rx_with_conditions))
rx_with_conditions[["DUPERSID", "RXDRGNAM", "TC1S1", "ICD10CDX", "CCSR1X"]].head(5)

**Wrestle with this:** look at the head() above. Pick one row. In plain English: 'Patient X was filled drug Y (therapeutic class Z) and it's linked to condition W.' Walk through one row to make sure the merge did what you think.

---
## Section 7 — Merge with `is_chronic.xlsx` to flag chronic conditions

Your is_chronic file has a column `is_chronic` (1 = chronic, 0 = not chronic) for each 3-digit ICD-10 code. We merge it onto our frame using `ICD10CDX` as the join key.

**Doc:** [DataFrame.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html), `how="left"` keeps every row even if no match.

In [ ]:
# Keep only the columns we need from is_chronic.
is_chronic_subset = is_chronic[["ICD10CDX", "ICD10CDX_LABEL", "is_chronic"]].copy()
print("is_chronic_subset rows:", len(is_chronic_subset))
print("How many ICD-10 codes are flagged chronic?", int(is_chronic_subset["is_chronic"].sum()))

# Merge — left join so we keep all rx_with_conditions rows.
rx_with_chronic_flag = rx_with_conditions.merge(
    is_chronic_subset,
    on="ICD10CDX",
    how="left",
)

print("rx_with_chronic_flag rows:", len(rx_with_chronic_flag))
print("Rows where is_chronic == 1 (chronic):", int((rx_with_chronic_flag["is_chronic"] == 1).sum()))
print("Rows where is_chronic == 0 (not chronic):", int((rx_with_chronic_flag["is_chronic"] == 0).sum()))
print("Rows where is_chronic is NaN (ICD-10 not in your file):", int(rx_with_chronic_flag["is_chronic"].isna().sum()))

**Wrestle with this:** what does it mean when `is_chronic` is NaN after this merge? Should those rows be in your adherence analysis? Defend your answer.

(Hint: NaN means the ICD-10 code on this fill's linked condition didn't appear in your is_chronic file. Either the code is novel/rare and didn't make Appendix 1 Table 1, or it's the `'-15'` masked-code rows.)

---
## Section 8 — Filter to chronic conditions AND maintenance drugs

Two filters now:
1. `is_chronic == 1` — the linked condition is a chronic one.
2. `TC1S1` is in the maintenance-drug class set — the drug is taken on a schedule, not PRN.

The maintenance set (verified against the actual 2023 file — the codebook PDF has some old codes that aren't in this year's data):

**Year invariance:** the Multum `TC1S1` codes are stable across 2020/2021/2022/2023 for the classes we care about. However, the **fill counts per class will shift** — for example, antidiabetic (`99`) is much larger in 2023 than in 2020 because of the GLP-1 boom (semaglutide / Ozempic / Wegovy entering the market). Note any class whose count drops by more than 50% across years; that may be a Multum re-classification, not a real prescribing change.

In [ ]:
# Maintenance drug classes from Multum TC1S1, verified in 2023 MEPS.
# Key = TC1S1 code, Value = human-readable class name.
MAINTENANCE = {
    19:  "Statins",
    99:  "Antidiabetic",
    42:  "ACE inhibitors",
    56:  "ARBs",
    47:  "Beta blockers",
    48:  "Calcium channel blockers",
    49:  "Diuretics",
    103: "Thyroid hormones",
    125: "Inhalers / bronchodilators",
    249: "Antidepressants",
    272: "Proton pump inhibitors",
    64:  "Anticonvulsants / mood stabilizers",
}

# Print so it's visible in the notebook record.
for code_num, name in MAINTENANCE.items():
    print("  TC1S1 =", code_num, "->", name)

In [ ]:
# Two filters applied step by step.

# Filter 1: chronic condition.
is_chronic_condition = rx_with_chronic_flag["is_chronic"] == 1
chronic_rx = rx_with_chronic_flag.loc[is_chronic_condition].copy()
print("After chronic-condition filter:", len(chronic_rx))

# Filter 2: maintenance drug class.
is_maintenance_class = chronic_rx["TC1S1"].isin(list(MAINTENANCE.keys()))
chronic_maint_rx = chronic_rx.loc[is_maintenance_class].copy()
print("After maintenance-class filter:", len(chronic_maint_rx))

# Add a readable class name column.
chronic_maint_rx["DRUG_CLASS"] = chronic_maint_rx["TC1S1"].map(MAINTENANCE)

# How many fills per class?
print()
print("Fills per drug class:")
print(chronic_maint_rx["DRUG_CLASS"].value_counts())

**Wrestle with this:** the antidiabetic class has more fills than the statin class in 2023, even though statins are usually the most-prescribed chronic class in the U.S. What's different about 2023?

(Hint: GLP-1 agonists like semaglutide — Ozempic / Wegovy.)

---
## Section 9 — Find the index round, then compute eligible days

Here's the new concept. Stay with me.

Imagine two patients:
- **Patient A** has been on metformin all of 2023. She filled it in every round. Her *eligible window* is the full year, so the denominator for her adherence ratio should be 365.
- **Patient B** was diagnosed with diabetes in October 2023 and got her first metformin fill in November. Her *eligible window* is only the last two months. If we use 365 as the denominator for her too, she looks non-adherent even if she filled perfectly within her actual window.

Solution: use the round of first fill as the **index round** — the earliest point we know the patient was on the drug — and compute eligible days from there.

MEPS doesn't have calendar dates, only `PURCHRD` (round 1–5). Each round is about 4 months. We use round midpoints as approximate days:

| PURCHRD | Approx calendar window | Midpoint day (1 = Jan 1) |
|---|---|---|
| 1 or 3 | Jan – Apr | 60 |
| 2 or 4 | May – Aug | 180 |
| 3 or 5 | Sep – Dec | 300 |

**Doc:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [GroupBy.min](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.min.html), [Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)

**Year invariance:** the PURCHRD-to-day midpoint mapping is the same shape every year (rounds 1–5, ~4 months each), but the absolute calendar dates shift slightly because MEPS fielding moves. For our purposes the approximation is identical. `compute_eligible_days(maintenance_rx)` is the eventual function — no year argument required.

In [ ]:
# Step 9.1: find the first PURCHRD per (DUPERSID, DRUG_CLASS).
index_round = (
    chronic_maint_rx
    .groupby(["DUPERSID", "DRUG_CLASS"])["PURCHRD"]
    .min()
    .reset_index()
)
index_round = index_round.rename(columns={"PURCHRD": "index_round"})

print("Number of unique (person, drug class) pairs:", len(index_round))
print()
print("Distribution of index rounds:")
print(index_round["index_round"].value_counts().sort_index())
index_round.head(5)

In [ ]:
# Step 9.2: map each index round to its approximate midpoint day.
# Build the mapping as a plain dictionary.
ROUND_TO_DAY = {
    1: 60,
    2: 180,
    3: 300,
    4: 180,
    5: 300,
}

index_round["index_day"] = index_round["index_round"].map(ROUND_TO_DAY)

# Eligible days = 365 minus index_day plus 1 (so day 60 to day 365 inclusive is 306 days).
index_round["eligible_days"] = 365 - index_round["index_day"] + 1

print("Eligible-days summary:")
print(index_round["eligible_days"].describe())
index_round.head(5)

**Wrestle with this:** is `365 - midpoint + 1` the right formula, or should it be `365 - midpoint`? What's the difference and does it matter for the headline result?

(Spoiler: the median PDC by class doesn't change materially either way. But you should be able to defend which version you chose.)

---
## Section 10 — Sum days covered, merge in eligible days, compute PDC

**The PDC formula:** PDC = sum(days supplied) / eligible_days, capped at 1.0.

Capping at 1.0 prevents stockpilers — patients who refill before the previous fill runs out — from showing > 100% adherence. This is the standard convention.

**Doc:** [GroupBy.sum](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.sum.html), [Series.clip](https://pandas.pydata.org/docs/reference/api/pandas.Series.clip.html)

**Year invariance:** the PDC formula is identical for any year. `compute_pdc(maintenance_rx)` is the function. When you call it on 2020 maintenance_rx vs 2023 maintenance_rx, the only difference in output is the data; the code is the same.

In [ ]:
# Step 10.1: sum days supplied per (DUPERSID, DRUG_CLASS).
days_covered_per_pair = (
    chronic_maint_rx
    .groupby(["DUPERSID", "DRUG_CLASS"])["RXDAYSUP"]
    .sum()
    .reset_index()
)
days_covered_per_pair = days_covered_per_pair.rename(columns={"RXDAYSUP": "days_covered"})

print("Days-covered table shape:", days_covered_per_pair.shape)
days_covered_per_pair.head(5)

In [ ]:
# Step 10.2: merge with the index_round / eligible_days frame.
pdc = days_covered_per_pair.merge(
    index_round,
    on=["DUPERSID", "DRUG_CLASS"],
    how="inner",
)

print("After merge:", pdc.shape)
pdc.head(5)

In [ ]:
# Step 10.3: compute the ratio.
pdc["PDC_raw"] = pdc["days_covered"] / pdc["eligible_days"]

# Cap at 1.0.
pdc["PDC"] = pdc["PDC_raw"].clip(upper=1.0)

print("PDC summary (capped at 1.0):")
print(pdc["PDC"].describe())

# How many got capped?
n_capped = (pdc["PDC_raw"] > 1.0).sum()
pct_capped = 100 * n_capped / len(pdc)
print()
print("Pairs with PDC_raw > 1.0 (capped to 1.0):", n_capped, "(", round(pct_capped, 1), "%)")

**Wrestle with this:** the fraction capped is large. What kind of patient ends up with raw PDC > 1.0? Three possibilities — pick one and defend.

1. Stockpilers — patients who refill before they run out, building a buffer.
2. Mail-order 90-day fills where one fill of 90 days for someone with a 6-month observation window comes out fine but a second 90-day fill in the same window overshoots.
3. Patients who have prevalent therapy AND we mis-estimated their eligible window (because they actually started before 2023).

---
## Section 11 — Sub-question 1, view 1: PDC by drug class

Now we answer Sub-question 1 the first way: which medication classes show lower refill continuity?

**Doc:** [seaborn.boxplot](https://seaborn.pydata.org/generated/seaborn.boxplot.html), [matplotlib.pyplot.axvline](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axvline.html)

**For the all-year version (`allYearMerge_v2.ipynb`):** stack the per-year PDC frames with a `meps_year` column, then either (a) plot one boxplot per year (faceted), (b) plot a single 4-year aggregated boxplot per class, or (c) plot trajectory of median PDC per class across years. Mehak will ask which view tells the story — pick one and defend.

In [ ]:
# Order classes by median PDC (ascending = least adherent first).
median_per_class = pdc.groupby("DRUG_CLASS")["PDC"].median().sort_values()
class_order = median_per_class.index.tolist()

print("Class order (lowest median first):")
for class_name in class_order:
    print("  {:35s} median PDC = {:.2f}".format(class_name, median_per_class[class_name]))

In [ ]:
# Make the boxplot.
fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=pdc, y="DRUG_CLASS", x="PDC", order=class_order, ax=ax)
ax.axvline(0.75, ls="--", color="red", label="Your 0.75 threshold")
ax.axvline(0.80, ls="--", color="orange", label="CMS Star 0.80")
ax.set_title("PDC distribution per maintenance drug class — chronic patients, 2023 MEPS")
ax.set_xlabel("PDC (capped at 1.0)")
ax.set_ylabel("")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Numeric summary alongside the plot so the story has both the picture and the number.
summary_by_class = pdc.groupby("DRUG_CLASS")["PDC"].agg(["count", "median", "mean"])

# Compute percent adherent at each threshold with a simple for-loop (not lambdas).
pct_at_75 = []
pct_at_80 = []
for class_name in summary_by_class.index:
    subset = pdc.loc[pdc["DRUG_CLASS"] == class_name, "PDC"]
    pct_at_75.append(100 * (subset >= 0.75).mean())
    pct_at_80.append(100 * (subset >= 0.80).mean())

summary_by_class["pct_adherent_075"] = pct_at_75
summary_by_class["pct_adherent_080"] = pct_at_80
summary_by_class = summary_by_class.sort_values("median")
print(summary_by_class.round(2))

**Wrestle with this:**
1. Which class has the lowest median PDC? Is that a real adherence finding, or is the class low because more of its patients are *incident* in 2023 (just started therapy)?
2. Which class has the highest? Why — chronicity, dosing simplicity, demographics?
3. Pick one class you would defend as a real finding to an interviewer, and one you would explicitly say you don't trust yet. Write a sentence for each.

---
## Section 12 — Sub-question 1, view 2: PDC by condition (ICD-10)

Same data, different rollup. Group by `ICD10CDX` instead of drug class. This tells you whether adherence differs by what the patient is being treated for.

We need to bring the condition column into the PDC frame. Since we merged it in earlier and then collapsed to (DUPERSID, DRUG_CLASS), it got dropped. So we rejoin from the pre-aggregation frame.

**Doc:** [DataFrame.drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

**For the all-year version:** same code, run per year, stack the outputs with a `meps_year` column. The most-common-condition logic is year-independent.

In [ ]:
# Get the (DUPERSID, DRUG_CLASS) → primary ICD10CDX mapping.
# Many fills per pair share the same condition, but a few have multiple.
# For this view, take the most common ICD10CDX per (person, class).
primary_condition = (
    chronic_maint_rx
    .groupby(["DUPERSID", "DRUG_CLASS", "ICD10CDX", "ICD10CDX_LABEL"])
    .size()
    .reset_index()
    .rename(columns={0: "n_fills"})
)
# Sort so the most frequent (ICD10CDX per (DUPERSID, DRUG_CLASS)) is on top, keep first.
primary_condition = primary_condition.sort_values(
    ["DUPERSID", "DRUG_CLASS", "n_fills"],
    ascending=[True, True, False],
)
primary_condition = primary_condition.drop_duplicates(subset=["DUPERSID", "DRUG_CLASS"]).copy()

print("Primary-condition rows:", len(primary_condition))
primary_condition.head(5)

In [ ]:
# Merge into pdc.
pdc_with_cond = pdc.merge(
    primary_condition[["DUPERSID", "DRUG_CLASS", "ICD10CDX", "ICD10CDX_LABEL"]],
    on=["DUPERSID", "DRUG_CLASS"],
    how="inner",
)
print("pdc_with_cond rows:", len(pdc_with_cond))

# Group by condition.
summary_by_condition = pdc_with_cond.groupby(["ICD10CDX", "ICD10CDX_LABEL"])["PDC"].agg(["count", "median", "mean"])

# Keep only conditions with at least 50 (person, class) pairs — otherwise the percentile estimates are noisy.
summary_by_condition = summary_by_condition[summary_by_condition["count"] >= 50]

# Compute percent adherent.
pct_at_75_cond = []
for icd, label in summary_by_condition.index:
    subset = pdc_with_cond.loc[pdc_with_cond["ICD10CDX"] == icd, "PDC"]
    pct_at_75_cond.append(100 * (subset >= 0.75).mean())
summary_by_condition["pct_adherent_075"] = pct_at_75_cond

summary_by_condition = summary_by_condition.sort_values("median")
print(summary_by_condition.round(2))

In [ ]:
# Plot the top conditions.
# Use ICD10CDX_LABEL for readability; if it's too long, the plot will be hard to read.
plot_data = pdc_with_cond.merge(
    summary_by_condition.reset_index()[["ICD10CDX", "median"]],
    on="ICD10CDX",
)

condition_order = (
    summary_by_condition.reset_index()
    .sort_values("median")["ICD10CDX_LABEL"]
    .tolist()
)

fig, ax = plt.subplots(figsize=(11, max(4, 0.4 * len(condition_order))))
sns.boxplot(data=plot_data, y="ICD10CDX_LABEL", x="PDC", order=condition_order, ax=ax)
ax.axvline(0.75, ls="--", color="red", label="0.75 threshold")
ax.set_title("PDC distribution per condition (ICD-10 3-digit) — chronic patients, 2023 MEPS")
ax.set_xlabel("PDC (capped at 1.0)")
ax.set_ylabel("")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Wrestle with this:**
1. Compare the by-class plot to the by-condition plot. Do they tell the same story? Where do they disagree?
2. The condition that ends up lowest median — is the same patient population also showing up as lowest in the by-class view? If yes, that's signal. If no, why?
3. Pick one finding from either view that you would NOT defend in an interview (because the count is too small, or the patients overlap, or the cohort is too narrow). Say why.

---
## Section 15 — Sub-question 2a: does cost affect adherence?

**STUB — we don't code this today. The markdown is here so the structure of the notebook stays end-to-end and we know where to put the code next session.**

**What we're trying to answer:** within a drug class (start with statins or antidiabetic, then expand), does adherence drop as out-of-pocket cost goes up?

**MEPS columns we'll need:**
- `RXSF23X` on h248a — self/family OOP **per fill**. Median when positive: $9.90.
- `RXSLF23` on h251 — total self/family Rx OOP **per year** (NOT `RXSF23` — that name doesn't exist on h251).
- `DLAYPM42` on h251 — SAQ item literally asking 'did you delay a prescription because of cost in the last 12 months?' Values: 1 = yes, 2 = no, plus the standard sentinels. **Round 4/2 only, so SAQELIG = 1 must be true.**
- Sibling columns: `DLAYCA42` (delayed medical care), `DLAYDN42` (delayed dental).

**Plan for the code (next session):**
1. Load h251. Pull `DUPERSID`, `RXSLF23`, `RXTOT23`, `DLAYPM42`, `SAQELIG`, `POVCAT23`.
2. Clean sentinels (`< 0 → NaN`).
3. Merge h251 into your `pdc` frame on `DUPERSID`.
4. Bin `RXSLF23` into quintiles (use `pd.qcut`). Plot median PDC by OOP quintile within a class.
5. Cross-tab PDC by `DLAYPM42` (1 vs 2). Patients who said 'I delayed for cost' should have lower PDC if your model is picking up real signal.
6. Stratify by `POVCAT23` (poverty category 1–5) to see whether the OOP effect is concentrated in low-income patients.

**Forced-choice Mehak will ask:** 'DLAYPM42 = 1 patients have lower PDC than DLAYPM42 = 2. Pick: (a) this validates the cost story, (b) this is circular, (c) this proves causation. One of these is right.'

**Docs to read before next session:**
- [pandas.qcut](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html)
- [seaborn.pointplot](https://seaborn.pydata.org/generated/seaborn.pointplot.html) or [seaborn.barplot](https://seaborn.pydata.org/generated/seaborn.barplot.html)

**Leakage check (your study plan W7):** `DLAYPM42` is asked in Round 4/2 of the same panel year. If your prediction window is **the same year**, this variable is leaky as a feature. If your prediction window is **the next year**, it's a fair retrospective feature. We make this decision explicitly when we get there.

---
## Section 16 — Sub-question 2b: how does chronic burden relate to adherence?

**STUB — code in a future session.**

**What we're trying to answer:** patients on more chronic conditions tend to have worse adherence on average. Does that hold in our data? Where does it break?

**MEPS columns we'll need (this is the big add):**

From **h251** (direct chronic flags, 1 = yes, 2 = no, negatives = missing):
- `DIABDX_M18` (diabetes), `HIBPDX` (hypertension), `CHOLDX` (high cholesterol)
- `CHDDX`, `ANGIDX`, `MIDX`, `OHRTDX` (coronary heart disease, angina, MI, other heart disease)
- `STRKDX` (stroke), `ASTHDX` (asthma), `EMPHDX` (emphysema)
- `ARTHDX` (arthritis), `CANCERDX` (cancer)
- `CHBRON31` (chronic bronchitis), `ADHDADDX` (ADHD/ADD)

From **h249** (for conditions not on h251):
- `N18` — CKD
- `I48` — atrial fibrillation
- `I50` — heart failure
- `E03`/`E04`/`E06`/`E07` — hypothyroidism
- `F32`/`F33` — depression
- `G20` — Parkinson's
- `G30` — Alzheimer's

**Plan for the code (next session):**
1. For each person on h251, count the number of `*DX` flags = 1.
2. Build a parallel count from h249 for the conditions not on h251 (using your `is_chronic.xlsx` to scope which ICD-10 codes count).
3. Combine into `chronic_burden_count` (total chronic conditions per person).
4. Merge into the `pdc` frame on `DUPERSID`.
5. Plot median PDC by `chronic_burden_count`, stratified by `DRUG_CLASS`.

**Forced-choice Mehak will ask:** 'Patients with 5+ chronic conditions have lower PDC than 1-condition patients in your data. Real signal or sample-size noise? How would you tell?'

**Honest expectation:** the population-level trend (more conditions → worse adherence) is well-established in the literature. The interesting question is per-class — does it hold for statins as strongly as for inhalers? Does it reverse for thyroid (where high-burden patients may be more medically engaged)?

**Doc to read:**
- [seaborn.violinplot](https://seaborn.pydata.org/generated/seaborn.violinplot.html) for stratified distributions

---
## Section 17 — Sub-question 2c: how do side effects relate to adherence?

**STUB — this is the one where MEPS hits a data wall. We'll talk through the honest answer when we get here.**

**What we're trying to answer:** patients who experience side effects from a medication are more likely to skip doses or discontinue. Does our data show this?

**The honest data picture:** MEPS does not have a 'side effects' column. There is no direct survey question that asks 'did you experience side effects from drug X?' This is the limit of household-survey data.

**What we have as proxies:**
- `RTHLTH31`, `RTHLTH42`, `RTHLTH53` — self-reported general health per round (1 = Excellent ... 5 = Poor). A decline across rounds could be illness progression OR side-effect-driven quality-of-life drop.
- `MNHLTH31`, `MNHLTH42`, `MNHLTH53` — self-reported mental health per round. Useful for antidepressant cohorts.
- `K6SUM42` (Kessler-6 distress) and `PHQ242` (PHQ-2 depression screen) — SAQ items. One round only.
- Discontinuation patterns inferred from your own data: a patient who fills the drug then has a gap of 60+ days then no further fills could be a discontinuation, which could be side-effect-driven (or unrelated — cost, life change, switched drug).

**Plan for the code (future session, if we get to it):**
1. Merge `RTHLTH31/42/53` and `MNHLTH31/42/53` into your `pdc` frame.
2. For each (person, drug class), compute the change in self-reported health from the round before the index round to the round after. If health declines while on a new drug, that's a signal worth flagging.
3. Compute a `discontinuation_flag`: 1 if the patient had at least one fill, then a 60+ day implied gap, then no further fills in that year. Use it as a poor-man's side-effect signal.

**Defense answer for August:** 'MEPS does not directly capture side effects. We use perceived-health change across rounds and discontinuation patterns as imperfect proxies. A real side-effects study would need FAERS, EHR clinical notes, or a patient registry like CMS adverse-event reporting. The methodology in this notebook would transfer to those datasets without structural change.'

**Don't pretend to answer this question fully. State the limit honestly.** That's the August skill.

---
## Section 18 — Sub-question 3: can we predict future non-adherence?

**STUB — coded in your study plan's Weekend 7 (modeling table) and Weekend 8 (training + evaluation).**

**What we're trying to answer:** given what we know about a `(patient, drug class)` pair at the end of year T−1, can we predict whether their PDC in year T will be below the 0.75 threshold?

**The label (the thing we're predicting):**
- `label = PDC_in_year_T < 0.75` (1 = non-adherent in target year, 0 = adherent)

**The modeling table (one row per `(DUPERSID, DRUG_CLASS, target_year)`):**

Features from year T−1 ONLY (everything else leaks):
- `prior_year_PDC` — the PDC we just computed, but for T−1. Strongest predictor by far. The baseline-to-beat.
- `prior_year_RXSLF23` — total Rx OOP from h251.
- `prior_year_RXTOT23` — total Rx spend from h251.
- `chronic_burden_count` — from Section 15.
- `DLAYPM42_prior` — only from the prior panel year, to avoid leakage.
- `age_band_T_minus_1` — from h251 `AGE2YX` (year-specific).
- `sex`, `RACEV1X`, `POVCAT23_prior` — demographic basics.
- `insurance_type_T_minus_1` — `INSCOV2YX` end of T−1.
- `index_round_T_minus_1` — when the patient first appeared on the drug in T−1.
- `fill_count_T_minus_1` — pure utilization signal.

**Three baselines (must beat all three to claim the model has learned something):**
1. Random ranking — AUROC 0.5 floor.
2. Prior-year PDC alone — a one-feature logistic regression. If the full model doesn't beat this by ≥5 percentage points on AUROC, the new features added nothing.
3. Chronic-burden-count alone — clinical-intuition baseline.

**The metric (matched to the decision):**
Capacity-limited outreach team can call N patients per week. The honest metric is **precision-at-k** (precision in the top-K ranked patients). Not accuracy. Not AUROC alone. Both should be reported.

**Candidate models:**
- Logistic regression (interpretable baseline)
- Gradient boosting (XGBoost or LightGBM) for performance
- Compare against the baselines first; tune only if the model has already beaten the baselines.

**Train/test split:**
- Train on 2020 → 2021 → 2022 pairs.
- Test on the 2022 → 2023 pair (last year's prediction for this year).
- DO NOT random-split. Time-aware split or you're cheating.

**Plan for the code (Weekend 7–8):**
1. Build the modeling table from the all-year PDC frame in `allYearMerge_v2.ipynb`.
2. Train baselines.
3. Train the model.
4. Compute AUROC, AUPRC, precision-at-k, calibration (Brier score).
5. Error analysis: 5 patients the model got wrong. What do they have in common?

**Docs:**
- [scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [XGBoost Python API](https://xgboost.readthedocs.io/en/stable/python/python_api.html)
- [scikit-learn precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
- [scikit-learn TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)

---
## Section 13 — Summarize h250 to person-level and merge into the PDC frame

h250 is plan-level (one row per person × plan), but our PDC frame is at (person, drug class) level. We need to roll h250 up to person-level before merging. The features we want:

- `has_any_rx_coverage` — 1 if the person has at least one plan with `PMEDINS == 1`, else 0.
- `has_plan_without_rx` — 1 if any plan has `PMEDINS == 2` (some coverage but no Rx). Useful for the cost story.
- `mean_monthly_premium` — average of valid `OOPPREMX` across the person's plans. NaN if no plans have valid premium data.
- `max_deductible_bucket` — highest valid `ANNDEDCTP` across the person's plans. Higher = more out-of-pocket exposure.
- `has_hsa` — 1 if any plan has `HSAACCT == 1`.

**Year invariance:** h250 column names are the same every year (`PMEDINS`, `OOPPREMX`, `ANNDEDCTP`, `HSAACCT`). The summary code below works identically for 2020/2021/2022. **2023 is the last year h250 exists publicly.**

**Doc:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [Series.replace](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html)

In [ ]:
# Step 13.1: clean the h250 sentinel codes on the columns we use.
h250_clean = h250.copy()

# PMEDINS keeps 1 (yes) and 2 (no); negatives → NaN.
is_valid_pmedins = h250_clean["PMEDINS"].isin([1, 2])
h250_clean["PMEDINS_CLEAN"] = h250_clean["PMEDINS"].where(is_valid_pmedins, np.nan)

# OOPPREMX: positive values are valid premium dollars; negatives are NaN.
h250_clean["OOPPREMX_CLEAN"] = h250_clean["OOPPREMX"].where(h250_clean["OOPPREMX"] >= 0, np.nan)

# ANNDEDCTP: buckets 1-7 are valid; negatives are NaN.
h250_clean["ANNDEDCTP_CLEAN"] = h250_clean["ANNDEDCTP"].where(h250_clean["ANNDEDCTP"] >= 1, np.nan)

# HSAACCT: 1 (yes) and 2 (no); negatives → NaN.
is_valid_hsa = h250_clean["HSAACCT"].isin([1, 2])
h250_clean["HSAACCT_CLEAN"] = h250_clean["HSAACCT"].where(is_valid_hsa, np.nan)

print("After cleaning, valid PMEDINS rows:", int(h250_clean["PMEDINS_CLEAN"].notna().sum()))
print("After cleaning, valid OOPPREMX rows:", int(h250_clean["OOPPREMX_CLEAN"].notna().sum()))
print("After cleaning, valid ANNDEDCTP rows:", int(h250_clean["ANNDEDCTP_CLEAN"].notna().sum()))

In [ ]:
# Step 13.2: roll up to person level using a groupby on DUPERSID.
# We build the feature columns one at a time so it's explicit, not chained.

# Has any plan with Rx coverage?
person_has_rx = (
    h250_clean.groupby("DUPERSID")["PMEDINS_CLEAN"]
    .apply(lambda values: int((values == 1).any()))
    .reset_index(name="has_any_rx_coverage")
)

# Has any plan WITHOUT Rx coverage?
person_has_no_rx = (
    h250_clean.groupby("DUPERSID")["PMEDINS_CLEAN"]
    .apply(lambda values: int((values == 2).any()))
    .reset_index(name="has_plan_without_rx")
)

# Mean monthly premium across plans (skipping NaN).
person_mean_premium = (
    h250_clean.groupby("DUPERSID")["OOPPREMX_CLEAN"]
    .mean()
    .reset_index(name="mean_monthly_premium")
)

# Max deductible bucket.
person_max_deduct = (
    h250_clean.groupby("DUPERSID")["ANNDEDCTP_CLEAN"]
    .max()
    .reset_index(name="max_deductible_bucket")
)

# Has HSA?
person_has_hsa = (
    h250_clean.groupby("DUPERSID")["HSAACCT_CLEAN"]
    .apply(lambda values: int((values == 1).any()))
    .reset_index(name="has_hsa")
)

print("person_has_rx rows:", len(person_has_rx))
person_has_rx.head(5)

In [ ]:
# Step 13.3: merge the five person-level summaries into a single insurance frame.
insurance = person_has_rx.merge(person_has_no_rx, on="DUPERSID", how="outer")
insurance = insurance.merge(person_mean_premium, on="DUPERSID", how="outer")
insurance = insurance.merge(person_max_deduct, on="DUPERSID", how="outer")
insurance = insurance.merge(person_has_hsa, on="DUPERSID", how="outer")

print("Insurance summary rows:", len(insurance))
insurance.head(5)

In [ ]:
# Step 13.4: merge insurance into the PDC frame.
# Use a LEFT join so we keep all PDC rows even if the patient has no h250 record.
# Patients with no h250 record are people not on private insurance — usually Medicare or Medicaid.
pdc_with_insurance = pdc.merge(insurance, on="DUPERSID", how="left")

print("pdc_with_insurance rows:", len(pdc_with_insurance))
print()
print("How many PDC rows have a matching h250 record?")
print("  has_any_rx_coverage not null:",
      int(pdc_with_insurance["has_any_rx_coverage"].notna().sum()),
      "(", round(100 * pdc_with_insurance["has_any_rx_coverage"].notna().sum() / len(pdc_with_insurance), 1), "%)")
print()
print("Cross-tab: median PDC by has_any_rx_coverage")
print(pdc_with_insurance.groupby("has_any_rx_coverage", dropna=False)["PDC"].describe()[["count", "50%", "mean"]])

**Wrestle with this:** patients with `has_any_rx_coverage = NaN` are not on private insurance — they're typically Medicare-only, Medicaid-only, or uninsured. Their PDC distribution may look DIFFERENT from privately-insured patients (Medicare Part D works differently than private plan formularies). Pick: (a) drop them from this analysis, (b) keep them and report the NaN group separately, (c) impute them as 'has_any_rx_coverage = 1' because Medicare Part D covers Rx. Defend.

**For the all-year version:** h250 stops in 2023. For 2020/2021/2022 the same code works, but for any year after 2023 there is no h250. The script-port version uses `try/except FileNotFoundError` on the h250 load so the pipeline degrades gracefully when h250 isn't available.

---
## Section 14 — Save the PDC frames for next session

Save both views so next session you can layer cost (Sub-question 2a) and chronic burden (Sub-question 2b) without recomputing.

**Doc:** [DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)

**For the all-year version (`allYearMerge_v2.ipynb`):** the save pattern is the same, but the file names get a year suffix and you'll loop over years. The pattern is:

```python
for year in [2020, 2021, 2022, 2023]:
    rx, clnk, conditions = load_meps_year(year)
    rx_clean = clean_h248a(rx)
    clnk_pmed = filter_clnk_pmed(clnk)
    merged = merge_rx_with_conditions(rx_clean, clnk_pmed, conditions, is_chronic)
    maintenance = filter_chronic_maintenance(merged)
    pdc_year = compute_pdc(maintenance)
    pdc_year["meps_year"] = year
    pdc_year.to_csv(f"pdc_{year}_by_class.csv", index=False)

# Then stack all years for cross-year analysis:
all_pdc = pd.concat([pd.read_csv(f"pdc_{year}_by_class.csv") for year in [2020, 2021, 2022, 2023]])
all_pdc.to_csv("pdc_2020_to_2023.csv", index=False)
```

That loop IS the Python script we want by mid-July. Today's notebook is a hand-walked single iteration of it.

In [ ]:
# Save the PDC-by-(person, class) frame.
out_dir = Path.cwd()
pdc_path = out_dir / "pdc_2023_by_class.csv"
pdc.to_csv(pdc_path, index=False)
print("Saved", len(pdc), "rows to", pdc_path)

# Save the PDC-by-(person, class) + condition frame.
pdc_cond_path = out_dir / "pdc_2023_by_class_and_condition.csv"
pdc_with_cond.to_csv(pdc_cond_path, index=False)
print("Saved", len(pdc_with_cond), "rows to", pdc_cond_path)

# Save the PDC + insurance features frame for the cost analysis (Sub-question 2a).
pdc_ins_path = out_dir / "pdc_2023_with_insurance.csv"
pdc_with_insurance.to_csv(pdc_ins_path, index=False)
print("Saved", len(pdc_with_insurance), "rows to", pdc_ins_path)

---
## What this notebook added to what you already had

1. **A real denominator.** Your previous notebook gave you the numerator (sum of days supplied). This notebook gave you the denominator (eligible days from the index round) so the result is a ratio you can compare across patients.
2. **An eligibility window concept.** Patients who start mid-year don't get punished by a 365-day denominator anymore. The mapping is approximate (MEPS has no exact dates), but it's the honest version.
3. **A cleaner maintenance filter** by Multum subclass `TC1S1` (with the verified 2023 codes — the codebook PDF had a few wrong).
4. **The `DIABEQUIP = 1` and `RXDAYSUP = 999` cleanings** that your previous work glossed over.
5. **The full merge chain** — h248a → CLNK → h249 → is_chronic — done step-by-step so every cell is yours.
6. **Two views of Sub-question 1** — PDC by drug class AND PDC by condition.

## What's next

**Sub-question 2a (cost):** stratify PDC by `RXSLF23` quintile (annual self-pay OOP, lives on h251) and validate against `DLAYPM42` (the SAQ item that literally asks 'did you delay a prescription because of cost?').

**Sub-question 2b (chronic burden):** count the number of chronic conditions per patient from `h251` flags (DIABDX_M18, HIBPDX, CHOLDX, ...) plus `h249` ICD-10 (using your is_chronic.xlsx). Stratify PDC by burden count.

**Sub-question 2c (side effects):** MEPS doesn't directly have side-effects data. The honest answer uses perceived-health change across rounds (`RTHLTH31` → `RTHLTH42` → `RTHLTH53`) and gap-then-no-refill patterns as proxies. This is where you state a real data limitation.

**Sub-question 3 (prediction):** the modeling table is one row per `(DUPERSID, DRUG_CLASS, target_year)`, features from year T−1, label = `PDC_in_year_T < 0.75`. Baseline: prior-year PDC alone. Real model has to beat that.

## Three things to bring to the next session

1. The class (or condition) you would defend as a real finding from Section 11 or 12, and your one-sentence defense.
2. The class (or condition) you would NOT defend, and your one-sentence reason.
3. One question about the eligibility-window approximation in Section 9 that you couldn't fully answer to yourself yet. Don't guess — write it down.

---

## The path to Python scripts (early August)

Once you've worked through this notebook with Mehak, replicated it in your own 2023 notebook, extended it to all four years in `allYearMerge_v2.ipynb`, and coded Sub-questions 2a/2b/2c (Sections 15–17), the next move is to lift each section into a function in `src/med_adherence/`. We do this in early August, AFTER the modeling work in Section 18 has a coded prototype — that way the scripts include modeling from day one rather than getting rewritten.

The mapping is one-to-one with the sections above:

| Notebook section | Script function | Module |
|---|---|---|
| Section 1 | `load_meps_year(year)` (returns 5 DataFrames including h250) | `ingest.py` |
| Section 2 | `clean_h248a(df)` | `clean.py` |
| Section 3 | `exclude_diabequip(df)` | `clean.py` |
| Section 4 | `filter_clnk_pmed(clnk)` | `clean.py` |
| Sections 5–7 | `merge_rx_with_conditions(rx, clnk, conditions, is_chronic)` | `merge.py` |
| Section 8 | `filter_chronic_maintenance(merged, maintenance_codes)` | `cohort.py` |
| Sections 9–10 | `compute_pdc(maintenance_rx)` | `adherence.py` |
| Sections 11–12 | `plot_pdc_by_class(pdc)`, `plot_pdc_by_condition(pdc_with_cond)` | `viz.py` |
| Section 13 | `summarize_h250_to_person(h250)` | `insurance.py` |
| Sections 15–17 | `analyze_cost(pdc, h251)`, `analyze_burden(pdc, h251)`, `analyze_discontinuation(pdc, rx)` | `analysis.py` |
| Section 18 | `build_modeling_table(pdc_panel)`, `train_baselines(table)`, `train_model(table)` | `model.py` |

Once those exist, the orchestrator script is one short loop:

```python
# scripts/run_pipeline.py
from med_adherence.ingest import load_meps_year
from med_adherence.clean import clean_h248a, exclude_diabequip, filter_clnk_pmed
from med_adherence.merge import merge_rx_with_conditions
from med_adherence.cohort import filter_chronic_maintenance
from med_adherence.adherence import compute_pdc
from med_adherence.insurance import summarize_h250_to_person

all_pdc = []
for year in [2020, 2021, 2022, 2023]:
    rx, clnk, conditions, h250, is_chronic = load_meps_year(year)
    rx = clean_h248a(rx)
    rx = exclude_diabequip(rx)
    clnk = filter_clnk_pmed(clnk)
    merged = merge_rx_with_conditions(rx, clnk, conditions, is_chronic)
    maintenance = filter_chronic_maintenance(merged)
    pdc = compute_pdc(maintenance)
    insurance = summarize_h250_to_person(h250)
    pdc = pdc.merge(insurance, on='DUPERSID', how='left')
    pdc['meps_year'] = year
    all_pdc.append(pdc)

pd.concat(all_pdc).to_csv('reports/pdc_panel.csv', index=False)
```

Each function is testable in isolation. Each year's run is independent. When something breaks, you know which function to look at. This is the structure we move into in early August.